
# 01 — The corrected benchmark

Every model on one rolling-origin fold schedule, with naive baselines, fold-level
spreads, and Diebold–Mariano tests under FDR control.

Three things this fixes relative to the earlier study: models were compared across
*different* train/test splits; no naive baseline was ever computed; and on ~300 test
points the RMSE gaps between models sit inside fold-to-fold noise.

**Read this notebook together with 04.** At one step ahead the learned models beat a
one-line baseline only modestly, and that is the honest result here. The case for
learning is made at the lead times an allocator actually needs, which is notebook 04.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Works in Colab and locally. In Colab, clone the repo first:
#     !git clone <repo-url> bwalloc && %cd bwalloc
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

In [ ]:

from bwalloc.baselines import SeasonalNaive, standard_baselines
from bwalloc.data import load, sampling_profile
from bwalloc.evaluate import beats_baseline, dm_matrix, run_backtest, summarise
from bwalloc.features import FeatureConfig, build_features
from bwalloc.models import default_point_models
from bwalloc.splits import describe_folds, rolling_origin

OPERATOR = "gp"          # switch to "robi" and re-run
N_FOLDS = 8

df = load(OPERATOR)
profile = sampling_profile(df)
X, y = build_features(df, profile, FeatureConfig())
folds = rolling_origin(len(y), n_folds=N_FOLDS)
describe_folds(folds, X.index)

## Backtest

Every model and every baseline on identical folds.

In [ ]:

baselines = standard_baselines(y.to_numpy(), profile.daily_period)
baselines.append(SeasonalNaive(y.to_numpy(), period=24))   # the lag the original used

per_fold, predictions = run_backtest(
    X, y, folds, models=default_point_models(),
    baselines=baselines, season_lag=profile.daily_period,
)
summary = beats_baseline(summarise(per_fold))
summary[["model", "rmse_mean", "rmse_std", "mae_mean", "mase_mean",
         "vs_persistence", "beats_persistence"]]

**The gate.** A model that does not beat persistence on the same folds is not a result. `run_benchmark.py` exits non-zero if the top-ranked model fails this.

In [ ]:

best = summary.iloc[0]
assert bool(best["beats_persistence"]), f"{best['model']} does not beat persistence"
print(f"best: {best['model']} — RMSE {best['rmse_mean']:.3f} ± {best['rmse_std']:.3f} "
      f"({best['vs_persistence']:+.1%} vs persistence)")

## Is the ranking real?

Diebold–Mariano across folds, Benjamini–Hochberg corrected because ten models is 45 comparisons. Most adjacent pairs are not separable on this much data — which is itself the finding.

In [ ]:

dm = dm_matrix(predictions, horizon=1)
top = summary["model"].head(4).tolist()
dm[dm["model_a"].isin(top) & dm["model_b"].isin(top)][
    ["model_a", "model_b", "dm_stat", "p_value", "significant_fdr", "winner"]
]


## Feature ablation — a negative result, reported

`experiments/run_benchmark.py` runs the full ablation. Its conclusion is worth
stating plainly because it cuts against the correction:

**Correcting the feature design does not improve accuracy on either operator.** Tree
ensembles route around a mis-specified `lag_24` by leaning on `lag_1..3`, so the
sampling-rate error costs almost nothing in RMSE once a flexible model is used.

That does not make the correction pointless — it makes its value *interpretive*
rather than predictive. Every seasonal claim in the earlier study (STL, ACF, FFT,
"peak at f ≈ 0.0417 ⇒ 24-hour cycle") was stated on the wrong time axis, and the
seasonal-naive comparison in notebook 00 shows what that costs a model that cannot
route around it.

In [ ]:

ablation = pd.read_csv(RESULTS / f"ablation_{OPERATOR}.csv")
ablation